[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C04_AI_Agents_Course/07_orchestration/07_orchestration.ipynb)

# 07 · 多智能体与编排 —— Orchestrator-Workers / Critic / Majority Vote 动手实验

配套讲解：`07_讲解.html`。本 notebook 把讲解第 2–5、7 节的结论全部变成可运行的数字：

**学习目标**
1. 在同一个可分解任务上实现并对比 **单 agent 基线** vs **orchestrator-workers**（质量分 / token 帐单 / 调用次数）；
2. 给单 agent 输出加一轮 **critic-executor** 审查修订，观察"2 次额外调用买多少质量"；
3. 用 5 道事实多选题验证 **majority vote** 的统计效应（Condorcet 条件）；
4. 练习：plan 解析与 schema 校验、平票回退投票、分阶段归因报告。

**运行说明（CPU 可跑，三级回退）**
- 若设置了 `OPENAI_API_KEY` → 走 OpenAI API（`OPENAI_MODEL` 可选，默认 `gpt-4o-mini`）；
- 否则若 `USE_LOCAL_LLM=1` → 走本地 `Qwen/Qwen2.5-1.5B-Instruct`（**需下载约 3.1GB 权重**，CPU 可跑但慢，GPU/MPS 更快）；
- 否则 → **确定性 mock**：按调用 `tag` 返回预置回复（不读 prompt），**零依赖跑通全流程**并完整演示所有统计效应。

> mock 的预置回复刻意带有真实系统中常见的瑕疵（单 agent 输出格式违规、各投票 agent 在不同题上出错），
> 这样对比实验在没有任何模型的机器上也能复现讲解里的结论。token 数用 `len(text)//4` 估算，仅用于相对比较。

In [ ]:
import os, re, json
from collections import Counter

# ===================== 实验材料：约 300 词的内嵌短文 =====================
ARTICLE = '''In December 1947, three physicists at Bell Labs - John Bardeen, Walter Brattain, and
William Shockley - demonstrated the first working transistor, a small semiconductor device that
could amplify and switch electronic signals. Before the transistor, electronics depended on vacuum
tubes, which were bulky, fragile, power-hungry, and prone to burning out. The transistor solved all
of these problems at once: it was tiny, rugged, efficient, and cheap to manufacture at scale. For
this breakthrough, the three inventors shared the Nobel Prize in Physics in 1956.

Early transistors were made from germanium, but the industry soon shifted to silicon, which
tolerates higher temperatures and forms a stable oxide layer that simplifies manufacturing. In 1958
and 1959, Jack Kilby at Texas Instruments and Robert Noyce at Fairchild Semiconductor independently
developed the integrated circuit, which placed many transistors on a single chip of silicon. This
invention turned the transistor from a discrete component into the basic unit of computation.

In 1965, Gordon Moore observed that the number of transistors that could be placed on a chip
doubled roughly every two years, an observation later known as Moore's law. For half a century this
exponential trend held, driving the cost of computation down by many orders of magnitude. A modern
processor contains tens of billions of transistors, each with features measured in nanometers.

The consequences reach far beyond computing. Transistors enabled portable radios, hearing aids,
satellites, medical instruments, and eventually the smartphone industry. Economists regard the
transistor as one of the most important inventions of the twentieth century, because nearly every
modern product and service depends, directly or indirectly, on cheap and reliable semiconductor
devices. Understanding its history is therefore essential for understanding the modern world.'''

# ===================== 三级回退后端选择 =====================
BACKEND = "mock"
_client = None
_pipe = None
if os.environ.get("OPENAI_API_KEY"):
    try:
        from openai import OpenAI
        _client = OpenAI()
        BACKEND = "openai"
    except ImportError:
        print("检测到 OPENAI_API_KEY 但未安装 openai 包，继续尝试其他后端")
if BACKEND == "mock" and os.environ.get("USE_LOCAL_LLM") == "1":
    try:  # 需下载约 3.1GB；CPU 可跑（慢），GPU/MPS 更快
        from transformers import pipeline
        _pipe = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct",
                         device_map="auto", torch_dtype="auto")
        BACKEND = "local"
    except Exception as e:
        print("本地模型不可用，回退 mock:", e)

# ===================== mock 后端的确定性预置回复（按 tag 路由） =====================
_GOOD_SUMMARY = ("The transistor, invented at Bell Labs in 1947 by Bardeen, Brattain, and Shockley, "
                 "replaced fragile vacuum tubes and earned the 1956 Nobel Prize in Physics. "
                 "The shift to silicon and the invention of the integrated circuit made the transistor "
                 "the basic unit of computation. Moore's law then described decades of exponential "
                 "growth that put billions of transistors into every modern device.")
_GOOD_KEYWORDS = ["transistor", "Bell Labs", "vacuum tube", "silicon", "Moore's law"]
_GOOD_QUESTIONS = ["In which year was the first transistor demonstrated at Bell Labs?",
                   "Which prize did Bardeen, Brattain, and Shockley share in 1956?"]
_PLAN = {"subtasks": [
    {"id": 1, "type": "summary",   "instruction": "Summarize the article in exactly 3 sentences."},
    {"id": 2, "type": "keywords",  "instruction": "Extract exactly 5 keywords that appear verbatim in the article."},
    {"id": 3, "type": "questions", "instruction": "Write exactly 2 factual questions answerable from the article, each ending with a question mark."},
]}

_MOCK = {
    # 单 agent 输出刻意带 3 处 rubric 违规：4 句摘要 / 只有 4 个关键词且 1 个不在原文 / 第 2 问缺问号
    "single": json.dumps({
        "summary": ("The transistor was invented at Bell Labs in 1947 by Bardeen, Brattain, and Shockley. "
                    "It replaced bulky vacuum tubes in electronics. "
                    "It earned the inventors the 1956 Nobel Prize in Physics. "
                    "Today billions of transistors power nearly every modern device."),
        "keywords": ["transistor", "Bell Labs", "silicon", "artificial intelligence"],
        "questions": ["In which year was the transistor invented?", "Who invented the transistor"]}),
    # planner 输出带围栏与多余文字——真实模型就是这样说话的
    "planner": ("Here is my decomposition plan:\n```json\n" + json.dumps(_PLAN, indent=2)
                + "\n```\nEach worker should receive only its own instruction."),
    "worker:summary":   json.dumps({"summary": _GOOD_SUMMARY}),
    "worker:keywords":  json.dumps({"keywords": _GOOD_KEYWORDS}),
    "worker:questions": json.dumps({"questions": _GOOD_QUESTIONS}),
    "critic": ("Review against the rubric: (1) the summary has 4 sentences but exactly 3 are required; "
               "(2) only 4 keywords are given and 'artificial intelligence' does not appear in the article; "
               "(3) question 2 is missing a question mark. Please revise."),
    "revise": json.dumps({"summary": _GOOD_SUMMARY, "keywords": _GOOD_KEYWORDS, "questions": _GOOD_QUESTIONS}),
}
# 三个投票 agent 的预置答案：各错 1 题且错在不同题上（模拟低相关错误，Condorcet 前提成立）
_MOCK_MCQ = {0: ["B", "C", "A", "D", "B"],
             1: ["B", "C", "B", "D", "C"],
             2: ["B", "A", "A", "D", "C"]}

def _mock_response(tag):
    if tag.startswith("mcq:"):
        _, p, q = tag.split(":")
        return _MOCK_MCQ[int(p)][int(q)]
    return _MOCK.get(tag, "OK")

# ===================== 统一 LLM 入口 + token 记账 =====================
LEDGER = []

def approx_tokens(text):
    return max(1, len(text) // 4)

def llm_call(prompt, system="", tag=""):
    '''统一入口：openai / 本地 Qwen / 确定性 mock；每次调用记账（近似 token）。'''
    if BACKEND == "openai":
        msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": prompt}]
        resp = _client.chat.completions.create(model=os.environ.get("OPENAI_MODEL", "gpt-4o-mini"),
                                               messages=msgs, temperature=0)
        text = resp.choices[0].message.content
    elif BACKEND == "local":
        msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": prompt}]
        out = _pipe(msgs, max_new_tokens=400, do_sample=False)
        text = out[0]["generated_text"][-1]["content"]
    else:
        text = _mock_response(tag)
    LEDGER.append({"tag": tag,
                   "prompt_tokens": approx_tokens(system + prompt),
                   "completion_tokens": approx_tokens(text)})
    return text

def usage_since(mark):
    calls = LEDGER[mark:]
    return {"calls": len(calls),
            "total_tokens": sum(c["prompt_tokens"] + c["completion_tokens"] for c in calls)}

print(f"LLM 后端: {BACKEND}（mock = 确定性预置回复，零依赖可跑）")
print(f"文章长度: {len(ARTICLE.split())} 词 ≈ {approx_tokens(ARTICLE)} token")

## 1. 实验任务与程序可验证 rubric + 单 agent 基线

任务对一篇约 300 词的文章产出三个字段——**天然可分解为 3 个独立子任务**：

| 字段 | 要求 | 程序可验证项 |
|---|---|---|
| `summary` | 恰好 3 句摘要 | 句数（按 `.!?` 切分），偏离每句扣 0.25 |
| `keywords` | 恰好 5 个、逐字出现在原文的关键词 | 数量合规（权重 0.5）+ 原文覆盖率（权重 0.5） |
| `questions` | 恰好 2 个事实性问题，以问号结尾 | 数量（权重 0.5）+ 问号比例（权重 0.5） |

总分 = 三个子分的平均（0~1）。这是模块 06 讲过的 **程序可验证 rubric**：便宜、确定、可回归。
真实系统中它通常与 LLM-judge 互补——这里刻意只用程序判分，让对比不受 judge 噪声干扰。

**单 agent 基线**：一次调用同时做 (a)(b)(c)。

In [ ]:
def parse_json_loose(text):
    '''容错提取模型输出中的第一个 JSON 对象（容忍 ``` 围栏与多余说明文字）。'''
    m = re.search(r"\{.*\}", text, re.S)
    if m is None:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None

def split_sentences(text):
    return [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]

def grade_output(obj, article):
    '''程序可验证 rubric -> 各子分与总分（0~1）。'''
    if obj is None:
        return {"summary": 0.0, "keywords": 0.0, "questions": 0.0, "total": 0.0}
    art = " ".join(article.split()).lower()  # 归一化空白：跨换行的短语（如 vacuum tubes）也能匹配
    n_sent = len(split_sentences(str(obj.get("summary", ""))))
    s_sum = max(0.0, 1.0 - 0.25 * abs(n_sent - 3)) if n_sent > 0 else 0.0
    kws = obj.get("keywords") or []
    fmt = max(0.0, 1.0 - 0.2 * abs(len(kws) - 5))
    cov = sum(1 for k in kws if " ".join(str(k).split()).lower() in art) / len(kws) if kws else 0.0
    s_kw = 0.5 * fmt + 0.5 * cov
    qs = obj.get("questions") or []
    cnt = 1.0 if len(qs) == 2 else (0.5 if qs else 0.0)
    qmark = sum(1 for q in qs if str(q).strip().endswith("?")) / len(qs) if qs else 0.0
    s_q = 0.5 * cnt + 0.5 * qmark
    return {"summary": round(s_sum, 3), "keywords": round(s_kw, 3), "questions": round(s_q, 3),
            "total": round((s_sum + s_kw + s_q) / 3, 3)}

RUBRIC_TEXT = ("Produce a JSON object with fields: summary (exactly 3 sentences), "
               "keywords (exactly 5 keywords, each appearing verbatim in the article), "
               "questions (exactly 2 factual questions answerable from the article, "
               "each ending with a question mark).")

def run_single_agent(article):
    prompt = f"Article:\n{article}\n\nTask: {RUBRIC_TEXT}\nReturn ONLY the JSON object."
    return llm_call(prompt, system="You are a careful analyst.", tag="single")

mark = len(LEDGER)
single_raw = run_single_agent(ARTICLE)
single_obj = parse_json_loose(single_raw)
single_score = grade_output(single_obj, ARTICLE)
single_usage = usage_since(mark)

print("单 agent 输出:", json.dumps(single_obj, ensure_ascii=False, indent=2))
print("\n单 agent 得分:", single_score)
print("单 agent 用量:", single_usage)
# mock 模式下应看到三处扣分：摘要 4 句(0.75)、关键词缺 1 个且 1 个不在原文、问题缺问号

## 2. Orchestrator-Workers：分解 → 隔离执行 → 聚合

对应讲解第 3 节，三个阶段各落实一个设计决策：

1. **planner**（1 次调用）：把任务分解为子任务 JSON——`{"subtasks": [{"id", "type", "instruction"}]}`，
   规格必须详细（输出格式、数量约束），含糊规格是下游一切错误的源头 [Anthropic 2025]；
2. **3 个 worker**（各 1 次调用，**完全隔离的上下文**）：每个 worker 只看到文章 + 自己的子任务规格，
   看不到其他 worker 的存在——真实部署中这 3 个调用可以 `asyncio.gather` 并行（省墙钟，不省 token）；
3. **aggregator**（本实验用纯代码）：采用**专属所有权**冲突消解——每个字段只取自负责它的 worker，
   越权输出一律忽略，冲突在结构上不存在。

注意账单：文章这段**公共上下文被 planner + 每个 worker 各支付一次**，这就是讲解第 5 节 Cost 公式里
$T_{\text{shared}}$ 被支付 $k$ 次的具体体现。

In [ ]:
PLAN_SCHEMA_TYPES = {"summary", "keywords", "questions"}

def run_planner(article):
    prompt = (f"Article:\n{article}\n\nDecompose the following task into 3 independent subtasks, "
              f"one per output field. Task: {RUBRIC_TEXT}\n"
              'Return JSON: {"subtasks": [{"id": <int>, "type": "summary|keywords|questions", '
              '"instruction": "<detailed spec for the worker>"}]}')
    return llm_call(prompt, system="You are the PLANNER of a multi-agent system.", tag="planner")

def run_worker(article, subtask):
    # 上下文隔离：worker 只看到文章 + 自己的子任务规格
    prompt = (f"Article:\n{article}\n\nYour single subtask: {subtask['instruction']}\n"
              f"Return ONLY a JSON object with the single field '{subtask['type']}'.")
    sys = f"You are WORKER #{subtask['id']} in a multi-agent system. Do exactly your subtask, nothing else."
    return llm_call(prompt, system=sys, tag=f"worker:{subtask['type']}")

def aggregate(worker_objs):
    '''专属所有权聚合：每个字段只取自负责它的 worker。'''
    return {field: (worker_objs[field].get(field) if worker_objs.get(field) else None)
            for field in ["summary", "keywords", "questions"]}

mark = len(LEDGER)
plan_obj = parse_json_loose(run_planner(ARTICLE))
subtasks = plan_obj["subtasks"]
print(f"planner 产出 {len(subtasks)} 个子任务:")
for st in subtasks:
    print(f"  [{st['id']}] {st['type']}: {st['instruction']}")

worker_objs = {}
for st in subtasks:  # 真实部署中可并行
    worker_objs[st["type"]] = parse_json_loose(run_worker(ARTICLE, st))

multi_obj = aggregate(worker_objs)
multi_score = grade_output(multi_obj, ARTICLE)
multi_usage = usage_since(mark)

print("\n========== 单 vs 多 agent 对比 ==========")
print(f"{'系统':<24}{'质量分':>8}{'调用数':>8}{'总token':>10}")
for name, score, u in [("single-agent", single_score["total"], single_usage),
                       ("orchestrator-workers", multi_score["total"], multi_usage)]:
    print(f"{name:<24}{score:>8.3f}{u['calls']:>8}{u['total_tokens']:>10}")
ratio = multi_usage["total_tokens"] / single_usage["total_tokens"]
print(f"\n多 agent token 帐单 ≈ 单 agent 的 {ratio:.1f} 倍（公共上下文被重复支付 + 多次调用）")
print("质量分更高的代价是数倍 token——是否值得取决于任务价值（讲解第 1/5 节）。")

## 3. Critic-Executor 与 Majority Vote

**critic-executor**（讲解第 2 节模式④）：不重做任务，而是给单 agent 的草稿加一轮独立审查——
critic（独立上下文，只挑错不动手）列出 rubric 违规，executor 据此修订。只多 2 次调用，
是介于"单 agent"与"完整 orchestrator-workers"之间的性价比选项。

**majority vote**（讲解第 4 节）：5 道关于文章的事实多选题，3 个**持不同 system prompt** 的 agent
独立作答后多数表决，对比"单次回答"基线。Condorcet 公式
$P_{\text{maj}}(n,p)=\sum_{k\ge\lceil n/2\rceil}\binom{n}{k}p^k(1-p)^{n-k}$ 生效的前提是
$p>0.5$ **且错误近似独立**。mock 模式预置了"每个 agent 各错 1 题、且错在不同题上"的低相关噪声答案，
用来演示统计效应；真实 LLM 共享基座时错误高度相关，增益会显著缩水——这正是要点。

In [ ]:
def run_critic_revision(article, draft_raw):
    mark = len(LEDGER)
    critique = llm_call(f"Rubric: {RUBRIC_TEXT}\n\nDraft output:\n{draft_raw}\n\nList every rubric violation.",
                        system="You are a strict CRITIC. You only review, never rewrite.", tag="critic")
    revised_raw = llm_call((f"Article:\n{article}\n\nYour previous draft:\n{draft_raw}\n\n"
                            f"Critic feedback:\n{critique}\n\nReturn the corrected JSON only."),
                           system="You are the EXECUTOR. Fix every issue the critic found.", tag="revise")
    return critique, revised_raw, usage_since(mark)

critique, revised_raw, critic_usage = run_critic_revision(ARTICLE, single_raw)
revised_score = grade_output(parse_json_loose(revised_raw), ARTICLE)

print("critic 审查意见:\n ", critique)
print(f"\n质量分: 修订前 {single_score['total']:.3f} -> 修订后 {revised_score['total']:.3f}")
print(f"额外开销: {critic_usage['calls']} 次调用 / {critic_usage['total_tokens']} token")
print("\n注意：critic 能修掉的是『可被审查规则刻画』的缺陷（格式、数量、原文覆盖）；")
print("若 critic 与 executor 同源（同一基座模型），两者共享知识盲区，事实性幻觉未必能被发现。")

In [ ]:
MCQS = [
    {"q": "In which year was the first transistor demonstrated?",
     "options": {"A": "1937", "B": "1947", "C": "1958", "D": "1965"}, "answer": "B"},
    {"q": "Where was the transistor invented?",
     "options": {"A": "Texas Instruments", "B": "Fairchild Semiconductor", "C": "Bell Labs", "D": "MIT"},
     "answer": "C"},
    {"q": "Which prize did the three inventors share in 1956?",
     "options": {"A": "Nobel Prize in Physics", "B": "Nobel Prize in Chemistry", "C": "Turing Award", "D": "Fields Medal"},
     "answer": "A"},
    {"q": "The transistor primarily replaced which earlier component?",
     "options": {"A": "integrated circuit", "B": "capacitor", "C": "relay", "D": "vacuum tube"}, "answer": "D"},
    {"q": "Who observed that transistor counts double roughly every two years?",
     "options": {"A": "Robert Noyce", "B": "Jack Kilby", "C": "Gordon Moore", "D": "William Shockley"},
     "answer": "C"},
]
PERSONAS = [
    "You are a meticulous fact-checker. Answer strictly based on the article.",
    "You are a fast generalist. Answer from your overall knowledge.",
    "You are a skeptical reviewer. Double-check dates and names before answering.",
]

def ask_mcq(persona_idx, qid):
    q = MCQS[qid]
    opts = "\n".join(f"{k}. {v}" for k, v in q["options"].items())
    prompt = (f"Article:\n{ARTICLE}\n\nQuestion: {q['q']}\n{opts}\n"
              "Answer with a single letter (A/B/C/D).")
    raw = llm_call(prompt, system=PERSONAS[persona_idx], tag=f"mcq:{persona_idx}:{qid}")
    m = re.search(r"[ABCD]", raw)
    return m.group(0) if m else "A"

def simple_majority(answers):
    return Counter(answers).most_common(1)[0][0]

single_correct = vote_correct = 0
print(f"{'Q':<3}{'truth':<7}{'3 agent 答案':<20}{'vote':<6}{'single':<7}")
for qid, q in enumerate(MCQS):
    answers = [ask_mcq(p, qid) for p in range(len(PERSONAS))]
    voted = simple_majority(answers)
    single_ans = answers[0]  # 单次回答基线 = 第一个 agent
    vote_correct += int(voted == q["answer"])
    single_correct += int(single_ans == q["answer"])
    print(f"{qid:<3}{q['answer']:<7}{str(answers):<20}{voted:<6}{single_ans:<7}")

print(f"\n单次回答准确率: {single_correct}/{len(MCQS)}    3-agent 多数投票: {vote_correct}/{len(MCQS)}")
print("mock 预置答案中每个 agent 各错 1 题且错在不同题上（低相关错误）——投票把 3 个 4/5 聚合成 5/5。")
print("若三个 agent 在同一题上犯同样的错（高相关，同基座模型的常态），投票将毫无增益甚至放大错误。")

## ✏️ 练习 1：`plan_to_tasks` —— 健壮地解析并校验 planner 输出

讲解第 6 节的第一类失败是**分解错误传播**，工程上的第一道防线就是对 plan 做**解析容错 + schema 严格校验**：
解析要宽（模型输出常带围栏和闲话），校验要严（字段缺一不可，错了宁可立刻失败也不能放进 fan-out）。

实现 `plan_to_tasks(plan_json_text)`：
1. 容忍 ```` ```json ```` 围栏与 JSON 前后的多余说明文字；
2. 顶层必须有非空的 `subtasks` 列表；
3. 每个子任务必须同时含 `id` / `type` / `instruction` 三个字段，且 `type ∈ {summary, keywords, questions}`；
4. 任何一步失败 → `raise ValueError`（带可读的错误信息）；通过则返回 `list[dict]`。

**提示**：`re.search(r"\{.*\}", text, re.S)` 抓最外层 JSON 块；`json.loads` 失败时转抛 `ValueError`。10–20 行可完成。

In [ ]:
def plan_to_tasks(plan_json_text):
    '''解析 planner 原始文本 -> 校验后的子任务 list。校验失败 raise ValueError。'''
    # TODO: 1. 用正则从文本中抽出第一个 {...} JSON 块（提示：re.search + re.S）；找不到 -> ValueError
    # TODO: 2. json.loads；JSONDecodeError -> ValueError
    # TODO: 3. 校验 subtasks 为非空 list；每项含 id/type/instruction；type 合法
    # TODO: 4. 返回子任务 list
    raise NotImplementedError

In [ ]:
# ---- 练习 1 自测 ----
_clean = json.dumps({"subtasks": [{"id": 1, "type": "summary", "instruction": "3 sentences"}]})
_fenced = "Sure! Here is the plan:\n```json\n" + json.dumps(_PLAN) + "\n```\nLet me know."
_missing = json.dumps({"subtasks": [{"id": 1, "type": "summary"}]})  # 缺 instruction

t1 = plan_to_tasks(_clean)
assert isinstance(t1, list) and len(t1) == 1 and t1[0]["type"] == "summary"

t2 = plan_to_tasks(_fenced)
assert len(t2) == 3 and {t["type"] for t in t2} == {"summary", "keywords", "questions"}

try:
    plan_to_tasks(_missing)
    raise AssertionError("缺字段的 plan 应当 raise ValueError")
except ValueError:
    pass

print("✅ 练习 1 通过")

## ✏️ 练习 2：`majority_vote` —— 含平票回退策略的多数投票

第 3 节用的 `simple_majority` 在平票时行为未定义（`Counter.most_common` 的顺序不可靠）。
真实投票系统必须显式定义平票语义。实现 `majority_vote(answers, confidences=None)`：

- `answers` 为空 → 返回 `None`；
- 存在唯一最高票答案 → 返回它；
- **平票**且提供了 `confidences`（与 `answers` 等长，逐票置信度）→ 在平票候选中返回**置信度之和**最大者；
- 平票且无 `confidences` → 返回 `None`（弃权比瞎选更诚实——上游可借此触发重投或升级）。

**提示**：`Counter` 统计 → 取最高票数 → 筛出平票候选集合 → 按规则回退。15 行以内。

In [ ]:
def majority_vote(answers, confidences=None):
    '''多数投票；平票时按 confidences 之和回退，无 confidences 则返回 None。'''
    # TODO: 1. 空列表 -> None
    # TODO: 2. Counter 统计，找出得票最高的候选集合 tied
    # TODO: 3. len(tied) == 1 -> 直接返回
    # TODO: 4. 平票：有 confidences -> 返回 tied 中置信度之和最大者；否则 None
    raise NotImplementedError

In [ ]:
# ---- 练习 2 自测 ----
assert majority_vote([]) is None
assert majority_vote(["C"]) == "C"
assert majority_vote(["B", "B", "C"]) == "B"
assert majority_vote(["A", "B"]) is None                                  # 平票、无置信度 -> 弃权
assert majority_vote(["A", "B"], confidences=[0.4, 0.9]) == "B"           # 平票 -> 按置信度
assert majority_vote(["A", "B", "A", "B"], confidences=[0.9, 0.3, 0.2, 0.3]) == "A"  # A: 1.1 > B: 0.6
print("✅ 练习 2 通过")

## ✏️ 练习 3：`attribution_report` —— 分阶段归因报告

讲解第 7 节：端到端分数 ≈ $p_{\text{decomp}} \cdot \prod_i p_{\text{worker},i} \cdot p_{\text{agg}}$，
诊断的第一步是把各阶段质量分单独测出来、找到瓶颈。实现 `attribution_report(stage_scores)`，
输入形如 `{"decomposition": 0.9, "worker": 0.55, "aggregation": 0.8}`，返回 dict：

- `bottleneck`：分数最低的阶段；**平分时取流水线中更上游者**（上游错误向下游传播，代价更大）；
- `bottleneck_score`：该阶段的分数；
- `gap`：`max - min`（短板与长板的差距，gap 大 → 修瓶颈的收益大）；
- `healthy`：所有阶段分数 `>= 0.85` 时为 `True`；
- `advice`：一句针对瓶颈阶段的修复建议字符串（**必须包含瓶颈阶段名**，内容可参考讲解第 7 节的表格）。

**提示**：`min(STAGES, key=...)` 在平分时天然返回 `STAGES` 中更靠前的阶段。15 行以内。

In [ ]:
STAGES = ["decomposition", "worker", "aggregation"]  # 流水线顺序（上游 -> 下游）

def attribution_report(stage_scores):
    '''三阶段质量分 -> 瓶颈归因报告 dict（键：bottleneck / bottleneck_score / gap / healthy / advice）。'''
    # TODO: 1. 按 STAGES 顺序找最低分阶段（平分取最先出现者）
    # TODO: 2. 计算 gap = max - min；healthy = (min >= 0.85)
    # TODO: 3. 为瓶颈阶段生成包含阶段名的修复建议
    # TODO: 4. 组装并返回报告 dict
    raise NotImplementedError

In [ ]:
# ---- 练习 3 自测 ----
r = attribution_report({"decomposition": 0.9, "worker": 0.55, "aggregation": 0.8})
assert r["bottleneck"] == "worker" and abs(r["bottleneck_score"] - 0.55) < 1e-9
assert abs(r["gap"] - 0.35) < 1e-9 and r["healthy"] is False
assert "worker" in r["advice"]

r2 = attribution_report({"decomposition": 0.7, "worker": 0.7, "aggregation": 0.9})
assert r2["bottleneck"] == "decomposition"  # 平分 -> 取更上游阶段

r3 = attribution_report({"decomposition": 0.95, "worker": 0.9, "aggregation": 0.92})
assert r3["healthy"] is True and r3["bottleneck"] == "worker"

print("✅ 练习 3 通过")

## 📖 参考答案

先自己做，再对照。参考实现均可让上面的自测 cell 全部通过。

In [ ]:
# 练习 1 参考答案（先自己做，再对照）
def plan_to_tasks(plan_json_text):
    m = re.search(r"\{.*\}", plan_json_text, re.S)
    if m is None:
        raise ValueError("文本中找不到 JSON 对象")
    try:
        obj = json.loads(m.group(0))
    except json.JSONDecodeError as e:
        raise ValueError(f"JSON 解析失败: {e}")
    subtasks = obj.get("subtasks")
    if not isinstance(subtasks, list) or not subtasks:
        raise ValueError("缺少非空的 subtasks 列表")
    for st in subtasks:
        for field in ("id", "type", "instruction"):
            if field not in st:
                raise ValueError(f"子任务缺少字段: {field}")
        if st["type"] not in PLAN_SCHEMA_TYPES:
            raise ValueError(f"非法 type: {st['type']}")
    return subtasks

In [ ]:
# 练习 2 参考答案（先自己做，再对照）
def majority_vote(answers, confidences=None):
    if not answers:
        return None
    counts = Counter(answers)
    top = max(counts.values())
    tied = [a for a, c in counts.items() if c == top]
    if len(tied) == 1:
        return tied[0]
    if confidences is not None and len(confidences) == len(answers):
        conf_sum = {a: 0.0 for a in tied}
        for a, c in zip(answers, confidences):
            if a in conf_sum:
                conf_sum[a] += c
        return max(conf_sum, key=conf_sum.get)
    return None

In [ ]:
# 练习 3 参考答案（先自己做，再对照）
def attribution_report(stage_scores):
    bottleneck = min(STAGES, key=lambda s: stage_scores[s])  # min 平分时返回 STAGES 中更靠前者
    scores = [stage_scores[s] for s in STAGES]
    advice_map = {
        "decomposition": ("decomposition 是瓶颈：检查 plan 的覆盖率与冗余率，"
                          "给 planner 输出加 schema 校验，并把子任务规格写得更详细"),
        "worker": ("worker 是瓶颈：用 golden 子任务隔离测试各 worker，"
                   "检查幻觉率与格式合规率，必要时更换更强模型或补充 few-shot"),
        "aggregation": ("aggregation 是瓶颈：检查冲突消解策略与信息保真率，"
                        "让 worker 输出结构化、定长的结果以避免 lost-in-the-middle"),
    }
    return {"bottleneck": bottleneck,
            "bottleneck_score": stage_scores[bottleneck],
            "gap": max(scores) - min(scores),
            "healthy": min(scores) >= 0.85,
            "advice": advice_map[bottleneck]}

## 小结

| 实验 | 结论 |
|---|---|
| 单 vs orchestrator-workers | 多 agent 质量分更高，但 token 帐单数倍于单 agent——公共上下文被每个 worker 重复支付；并行只省墙钟不省钱 |
| critic-executor | 约 2 次额外调用即可修掉"可被审查规则刻画"的缺陷，是单/多 agent 之间的性价比选项；critic 与 executor 同源时共享盲区 |
| majority vote | 错误低相关 + 单体优于随机时，投票显著提分（Condorcet）；错误高相关时零增益甚至放大错误 |
| 三道练习 | plan 的容错解析 + 严格校验是分解错误传播的第一道防线；平票必须有显式回退语义；分阶段归因先于一切调优 |

**对评测工作者的提醒**：评估任何多 agent 系统时，永远要求三件套——端到端质量分、**与质量并列报告的成本**
[Kapoor 2024]、以及分阶段归因表（练习 3 的 `attribution_report` 就是最小原型）。只看端到端分数，
你无法区分"planner 不会切任务"和"worker 能力不足"，也无法判断 15 倍的 token 是否买来了等价的价值
[Anthropic 2025]。

**下一站 → 模块 08 · Agent 安全与监控**（`../08_agent_safety/08_讲解.html`）：多 agent 把信息转手次数
变多、来源追溯变难——被 prompt injection 攻陷的单个 worker 如何污染整个系统的输出，以及如何评测与防御。

---
## 🎯 真实数据胶囊题：真实 GSM8K 上的多 agent 投票编排

多 agent 编排最简单的形态是并行多个 agent + 多数投票。用真实 GSM8K 金标，模拟 M 个能力不同的 agent，验证多数投票编排优于最强单个 agent。

> 本模块新增的**真实数据**练习：自包含、用真实公开数据把本章方法跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, urllib.request, re
import numpy as np
CACHE=os.path.expanduser("~/.ai_agents_data"); os.makedirs(CACHE,exist_ok=True)
def _f(url,fn,headers=None):
    p=os.path.join(CACHE,fn)
    if not os.path.exists(p):
        req=urllib.request.Request(url, headers=headers or {})
        open(p,"wb").write(urllib.request.urlopen(req,timeout=40).read())
    return p
def gsm8k(n=300):
    p=_f("https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl","gsm8k_test.jsonl")
    return [json.loads(l) for l in open(p).read().splitlines()[:n]]
def gold(a): return a.split("####")[-1].strip().replace(",","")
def mbpp(n=100):
    p=_f("https://raw.githubusercontent.com/google-research/google-research/master/mbpp/mbpp.jsonl","mbpp.jsonl")
    return [json.loads(l) for l in open(p).read().splitlines()[:n]]

rows=gsm8k(150); golds=[gold(r["answer"]) for r in rows]
rng=np.random.default_rng(0)
abilities=[0.45,0.5,0.55,0.5,0.48]   # M 个 agent 各自单题正确率
def agent_answer(g, p):
    return g if rng.random()<p else str(int(rng.integers(0,99999)))

**练习**：实现 `ensemble_acc(golds, abilities, seed)`：每题让每个 agent 各答一次，多数投票，返回编排准确率。

In [ ]:
def ensemble_acc(golds, abilities, seed=0):
    # TODO: 每题对每个 ability 用 agent_answer 生成，多数投票，与 gold 比
    raise NotImplementedError


In [ ]:
# 自测
ens=ensemble_acc(golds, abilities, seed=1)
best_single=max(abilities)
assert ens > best_single, f"投票编排{ens:.2f}应优于最强单agent{best_single}"
print(f"多 agent 投票 {ens:.2f} > 最强单 agent {best_single} ✓")


### 📖 参考答案

In [ ]:
def ensemble_acc(golds, abilities, seed=0):
    from collections import Counter
    rng2=np.random.default_rng(seed); c=0
    for g in golds:
        votes=[g if rng2.random()<p else str(int(rng2.integers(0,99999))) for p in abilities]
        if Counter(votes).most_common(1)[0][0]==g: c+=1
    return c/len(golds)
print("✓ 并行+投票是多 agent 编排最稳的 baseline(本质是 ensemble)")